In [ ]:
# Installation des bibliothèques
!pip install -q numpy pandas xarray netCDF4 matplotlib seaborn scikit-learn shapely


In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from shapely.geometry import shape, Point
from shapely.ops import unary_union
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    ConfusionMatrixDisplay,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "data" / "97a482e22d0cb0b4273cb0928c26ced.nc"
GEO_PATH = BASE_DIR / "data" / "dz.json"

TARGET_VAR = "t2m"
TIME_NAME = "valid_time"
LAT_NAME = "latitude"
LON_NAME = "longitude"
TEST_SIZE = 0.2
N_CLASSES = 3
CLASS_NAMES = ["Froide", "Modérée", "Chaude"]

VAR_LABELS = {
    "t2m": "2m Temperature",
    "d2m": "2m Dewpoint Temperature",
    "tp": "Total Precipitation",
    "sp": "Surface Pressure",
    "ssr": "Surface Net Solar Radiation",
    "str": "Surface Net Thermal Radiation",
    "e": "Evaporation",
    "u10": "10m U-wind",
    "v10": "10m V-wind",
    "wind_speed": "Wind Speed",
}

REQUIRED_VARS = ["t2m", "d2m", "tp", "sp", "ssr", "str", "e", "u10", "v10"]


In [ ]:
def load_algeria_geometry(path):
    with open(path, 'r', encoding='utf-8') as file:
        geojson = json.load(file)

    geojson_type = geojson.get('type')
    if geojson_type == 'FeatureCollection':
        geometries = [
            shape(feature['geometry'])
            for feature in geojson.get('features', [])
            if feature.get('geometry') is not None
        ]
    elif geojson_type == 'Feature':
        geometries = [shape(geojson['geometry'])]
    else:
        geometries = [shape(geojson)]

    geometries = [
        geometry for geometry in geometries
        if geometry is not None and not geometry.is_empty
    ]
    if not geometries:
        raise ValueError('Aucune géométrie valide trouvée dans le fichier GeoJSON.')
    return unary_union(geometries)

algeria_geometry = load_algeria_geometry(GEO_PATH)
print('Géométrie Algérie chargée:', type(algeria_geometry).__name__)

coarse_ds = xr.open_dataset(DATA_PATH)
coarse_ds = coarse_ds.rename({
    'valid_time': TIME_NAME,
    'latitude': LAT_NAME,
    'longitude': LON_NAME,
})

for var in REQUIRED_VARS:
    if var not in coarse_ds.data_vars:
        raise ValueError(f'Variable manquante: {var}')

print(coarse_ds)


In [ ]:
inside_cells = coarse_ds[TARGET_VAR].notnull().any(dim=TIME_NAME)

overview_rows = []
for variable in REQUIRED_VARS + ['wind_speed']:
    if variable in coarse_ds.data_vars:
        da = coarse_ds[variable]
    elif variable == 'wind_speed':
        wind_speed = np.hypot(coarse_ds['u10'], coarse_ds['v10'])
        da = xr.DataArray(
            wind_speed,
            coords=coarse_ds['u10'].coords,
            dims=coarse_ds['u10'].dims,
            name='wind_speed',
        )
    else:
        continue

    valid_area = inside_cells.broadcast_like(da)
    denominator = int(valid_area.sum().item())
    missing_count = int((da.isnull() & valid_area).sum().item())
    values = da.where(inside_cells).values
    valid_values = values[np.isfinite(values)]

    overview_rows.append({
        'Variable': variable,
        'Description': VAR_LABELS.get(variable, variable),
        'Units': da.attrs.get('units', 'n/a'),
        'Mean': float(np.nanmean(valid_values)) if valid_values.size else np.nan,
        'Std': float(np.nanstd(valid_values)) if valid_values.size else np.nan,
        'Min': float(np.nanmin(valid_values)) if valid_values.size else np.nan,
        'Max': float(np.nanmax(valid_values)) if valid_values.size else np.nan,
        'Missing Cells': missing_count,
        'Coverage %': (1 - missing_count / denominator) * 100 if denominator else np.nan,
    })

overview = pd.DataFrame(overview_rows)
display(overview.round(4))


In [ ]:
spatial_weights = np.cos(np.deg2rad(coarse_ds[LAT_NAME]))
monthly_xarray = coarse_ds[REQUIRED_VARS + ['u10', 'v10']].copy()
monthly_xarray['wind_speed'] = np.hypot(monthly_xarray['u10'], monthly_xarray['v10'])
monthly_xarray = monthly_xarray.weighted(spatial_weights).mean(dim=[LAT_NAME, LON_NAME], skipna=True)

monthly_summary = monthly_xarray.to_dataframe().drop(columns=['number', 'expver'], errors='ignore')
monthly_summary = monthly_summary.reset_index()
monthly_summary[TIME_NAME] = pd.to_datetime(monthly_summary[TIME_NAME])
monthly_summary['month'] = monthly_summary[TIME_NAME].dt.month
monthly_summary['year'] = monthly_summary[TIME_NAME].dt.year
monthly_summary = monthly_summary.set_index(TIME_NAME)
monthly_summary.index.name = 'date'

monthly_summary.head()


In [ ]:
eda_variables = ['t2m', 'd2m', 'tp', 'wind_speed', 'sp']
fig, axes = plt.subplots(2, len(eda_variables), figsize=(22, 9))
for index, variable in enumerate(eda_variables):
    data = monthly_summary[variable].dropna()
    sns.histplot(data, kde=True, color='steelblue', ax=axes[0, index])
    axes[0, index].set_title(f'Distribution - {VAR_LABELS.get(variable, variable)}')
    sns.boxplot(y=data, ax=axes[1, index], color='lightsteelblue')
    axes[1, index].set_title(f'Boxplot - {VAR_LABELS.get(variable, variable)}')

plt.tight_layout()
plt.show()


In [ ]:
monthly_summary = monthly_summary.copy()
monthly_summary['t2m_c'] = monthly_summary['t2m'] - 273.15
tp_units = str(coarse_ds['tp'].attrs.get('units', 'm')).lower()
precip_factor = 1000 if tp_units in {'m', 'meter', 'metre'} else 1
monthly_summary['tp_mm'] = monthly_summary['tp'] * precip_factor

fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)
axes[0].plot(monthly_summary.index, monthly_summary['t2m_c'], color='darkorange', linewidth=2)
axes[0].set_title('Monthly Mean Temperature - Algeria')
axes[0].set_ylabel('Temperature (°C)')

axes[1].plot(monthly_summary.index, monthly_summary['tp_mm'], color='dodgerblue', linewidth=2)
axes[1].set_title('Monthly Mean Precipitation - Algeria')
axes[1].set_ylabel('Precipitation (mm)')

axes[2].plot(monthly_summary.index, monthly_summary['wind_speed'], color='forestgreen', linewidth=2)
axes[2].set_title('Monthly Mean Wind Speed - Algeria')
axes[2].set_ylabel('Wind Speed (m/s)')
axes[2].set_xlabel('Date')

plt.tight_layout()
plt.show()


In [ ]:
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
fig, axes = plt.subplots(2, 2, figsize=(17, 12))
seasonal_variables = ['t2m_c', 'tp_mm', 'wind_speed', 'sp']
for ax, variable in zip(axes.flat, seasonal_variables):
    sns.boxplot(data=monthly_summary, x='month', y=variable, ax=ax)
    ax.set_xticks(range(12))
    ax.set_xticklabels(month_names, rotation=45)
    ax.set_title(f'Seasonality - {variable}')

plt.tight_layout()
plt.show()


In [ ]:
heatmap_data = monthly_summary.pivot_table(index='year', columns='month', values='t2m_c', aggfunc='mean').reindex(columns=range(1, 13))
heatmap_data.columns = month_names

plt.figure(figsize=(16, 8))
sns.heatmap(heatmap_data, cmap='RdYlBu_r', annot=True, fmt='.1f', linewidths=0.5, cbar_kws={'label': 'Temperature (°C)'})
plt.title('Seasonality Heatmap - Algeria Temperature')
plt.xlabel('Month')
plt.ylabel('Year')
plt.tight_layout()
plt.show()


In [ ]:
# Calcul de la vitesse du vent à 10 mètres (m/s)
coarse_ds["wind_speed"] = np.sqrt(
    coarse_ds["u10"]**2 + coarse_ds["v10"]**2
)

coarse_ds["wind_speed"].attrs = {
    "long_name": "10 metre wind speed",
    "units": "m/s"
}

print("Wind speed calculated successfully.")
print(coarse_ds["wind_speed"])

In [ ]:
map_variables = [
    ('t2m', 'Mean Temperature (°C)', 'RdYlBu_r'),
    ('tp', 'Mean Precipitation (mm)', 'YlGnBu'),
    ('wind_speed', 'Mean Wind Speed (m/s)', 'viridis'),
]

fig, axes = plt.subplots(1, 3, figsize=(21, 7))

for ax, (variable, title, cmap) in zip(axes, map_variables):

    # Moyenne temporelle
    mean_da = coarse_ds[variable].mean(dim=TIME_NAME)

    # Conversion des unités
    if variable == 't2m':
        mean_da = mean_da - 273.15

    elif variable == 'tp':
        tp_units = str(coarse_ds['tp'].attrs.get('units', 'm')).lower()

        if tp_units in ['m', 'meter', 'metre']:
            mean_da = mean_da * 1000

    # Coordonnées géographiques
    lat = mean_da[LAT_NAME].values
    lon = mean_da[LON_NAME].values
    values = mean_da.values

    # Création de la grille géographique
    lon_grid, lat_grid = np.meshgrid(lon, lat)

    # Masque géographique de l'Algérie
    points = [
        Point(float(x), float(y))
        for x, y in zip(lon_grid.ravel(), lat_grid.ravel())
    ]

    inside_mask = np.array([
        algeria_geometry.contains(pt)
        or algeria_geometry.touches(pt)
        for pt in points
    ]).reshape(len(lat), len(lon))

    masked_values = np.where(
        inside_mask, values, np.nan
    )

    # Affichage de la carte
    mesh = ax.pcolormesh(
        lon,
        lat,
        masked_values,
        shading='auto',
        cmap=cmap
    )

    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

    fig.colorbar(
        mesh,
        ax=ax,
        shrink=0.9,
        label=title.split('(')[-1].replace(')', '')
        if '(' in title else ''
    )

plt.suptitle(
    'Spatial Climate Distribution — Algeria (2015–2025)',
    fontsize=15,
    fontweight='bold'
)

plt.tight_layout()
plt.show()

In [ ]:
correlation_variables = [var for var in REQUIRED_VARS + ['wind_speed'] if var in monthly_summary.columns]
correlation_matrix = monthly_summary[correlation_variables].corr(method='pearson')

plt.figure(figsize=(14, 11))
sns.heatmap(correlation_matrix, cmap='coolwarm', center=0, vmin=-1, vmax=1, annot=True, fmt='.2f', linewidths=0.5, square=True)
plt.title('Correlation Matrix - Algeria Climate Variables')
plt.tight_layout()
plt.show()


In [ ]:
scatter_pairs = [('t2m', 'd2m'), ('t2m', 'tp'), ('t2m', 'ssr'), ('tp', 'e')]
scatter_pairs = [(x, y) for x, y in scatter_pairs if x in monthly_summary.columns and y in monthly_summary.columns]
fig, axes = plt.subplots(1, len(scatter_pairs), figsize=(6 * len(scatter_pairs), 5))
axes = np.atleast_1d(axes)
for ax, (x, y) in zip(axes, scatter_pairs):
    sns.scatterplot(data=monthly_summary, x=x, y=y, ax=ax, alpha=0.8)
    ax.set_title(f'{x} vs {y}')

plt.tight_layout()
plt.show()


In [ ]:
spatial_data = coarse_ds[REQUIRED_VARS + ['u10', 'v10']].to_dataframe().reset_index()
spatial_data['wind_speed'] = np.hypot(spatial_data['u10'], spatial_data['v10'])
spatial_data = spatial_data.dropna(subset=[TARGET_VAR]).copy()
spatial_data[TIME_NAME] = pd.to_datetime(spatial_data[TIME_NAME])
spatial_data = spatial_data.drop_duplicates(subset=[TIME_NAME, LAT_NAME, LON_NAME])
spatial_data = spatial_data.sort_values([LAT_NAME, LON_NAME, TIME_NAME]).reset_index(drop=True)
spatial_data['month'] = spatial_data[TIME_NAME].dt.month
spatial_data['year'] = spatial_data[TIME_NAME].dt.year

model_data = spatial_data.copy()
model_data['target_temp'] = model_data.groupby([LAT_NAME, LON_NAME])[TARGET_VAR].shift(-1)
model_data['target_date'] = model_data.groupby([LAT_NAME, LON_NAME])[TIME_NAME].shift(-1)
model_data = model_data.dropna(subset=['target_temp', 'target_date']).copy()

feature_cols = [col for col in REQUIRED_VARS + ['wind_speed', 'month', 'year'] if col in model_data.columns]
model_data = model_data[feature_cols + ['target_temp', 'target_date', TIME_NAME, LAT_NAME, LON_NAME]].copy()

model_data['temp_class'] = pd.cut(
    model_data['target_temp'],
    bins=np.linspace(model_data['target_temp'].min(), model_data['target_temp'].max(), N_CLASSES + 1),
    labels=range(N_CLASSES),
    include_lowest=True,
).astype(int)

missing_report = (model_data[feature_cols].isna().mean() * 100).reset_index()
missing_report.columns = ['Variable', 'Missing %']

outlier_rows = []
for col in feature_cols:
    series = model_data[col].dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = series[(series < lower) | (series > upper)]
    outlier_rows.append({'Variable': col, 'Outliers': int(outliers.shape[0])})

outlier_report = pd.DataFrame(outlier_rows)
display(missing_report.sort_values('Missing %', ascending=False))
display(outlier_report.sort_values('Outliers', ascending=False))


In [ ]:
unique_dates = np.sort(model_data[TIME_NAME].dropna().unique())
if len(unique_dates) < 6:
    raise ValueError('Nombre de mois insuffisant pour Train/Test.')

split_position = int(len(unique_dates) * (1 - TEST_SIZE))
split_position = min(max(split_position, 2), len(unique_dates) - 2)
split_date = pd.Timestamp(unique_dates[split_position])

train_data = model_data[model_data['target_date'] < split_date].copy()
test_data = model_data[model_data['target_date'] >= split_date].copy()

X_train = train_data[feature_cols].copy()
X_test = test_data[feature_cols].copy()
y_train = train_data['temp_class'].astype(int)
y_test = test_data['temp_class'].astype(int)

print(f'Train rows: {len(train_data)} | Test rows: {len(test_data)}')
print(f'Split date: {split_date}')


In [ ]:
def scaled_pipeline(estimator):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', estimator),
    ])

def unscaled_pipeline(estimator):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', estimator),
    ])

models = {
    'KNN': scaled_pipeline(KNeighborsClassifier(n_neighbors=7)),
    'SVM': scaled_pipeline(SVC(probability=True, kernel='rbf', C=10, gamma='scale', random_state=42)),
    'Random Forest': unscaled_pipeline(RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42)),
    'AdaBoost': unscaled_pipeline(AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=3), n_estimators=200, learning_rate=0.1, random_state=42)),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f'{name} trained.')

predictions = {name: model.predict(X_test) for name, model in models.items()}


In [ ]:
base_models = ['KNN', 'SVM', 'Random Forest']
base_prediction_matrix = np.column_stack([predictions[model_name] for model_name in base_models])
predictions['Hard Voting'] = np.array([
    np.bincount(row.astype(int), minlength=N_CLASSES).argmax()
    for row in base_prediction_matrix
])

probability_matrix = []
for model_name in base_models:
    model = models[model_name]
    probability_matrix.append(model.predict_proba(X_test))

mean_probabilities = np.mean(np.stack(probability_matrix), axis=0)
predictions['Soft Voting'] = mean_probabilities.argmax(axis=1)

print('Voting predictions computed.')


In [ ]:
results = []
for model_name, y_pred in predictions.items():
    results.append({
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision macro': precision_score(y_test, y_pred, average='macro', zero_division=0),
        'Recall macro': recall_score(y_test, y_pred, average='macro', zero_division=0),
        'F1 macro': f1_score(y_test, y_pred, average='macro', zero_division=0),
    })

supervised_results = pd.DataFrame(results).sort_values('F1 macro', ascending=False).reset_index(drop=True)
display(supervised_results)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 11))
for ax, (model_name, y_pred) in zip(axes.flat, predictions.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        y_pred,
        labels=list(range(N_CLASSES)),
        display_labels=CLASS_NAMES,
        cmap='Blues',
        colorbar=False,
        ax=ax,
    )
    ax.set_title(model_name)

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(11, 6))
sns.barplot(data=supervised_results, x='F1 macro', y='Model', color='steelblue')
plt.title('Model Comparison - F1 Macro')
plt.xlabel('F1 Score')
plt.ylabel('Machine Learning Model')
plt.xlim(0, 1)
plt.tight_layout()
plt.show()


In [ ]:
rf_model = models['Random Forest'].named_steps['model']
rf_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_,
}).sort_values('Importance', ascending=False)
display(rf_importance.round(4))

plt.figure(figsize=(11, 7))
sns.barplot(data=rf_importance.head(10), x='Importance', y='Feature', color='darkorange')
plt.title('Top 10 Importances - Random Forest')
plt.tight_layout()
plt.show()


In [ ]:
RESULTS_DIR = BASE_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

overview.to_csv(RESULTS_DIR / '01_dataset_overview.csv', index=False)
monthly_summary.to_csv(RESULTS_DIR / '02_descriptive_statistics.csv')
missing_report.to_csv(RESULTS_DIR / '03_missing_values.csv', index=False)
outlier_report.to_csv(RESULTS_DIR / '04_outliers_iqr.csv', index=False)
supervised_results.to_csv(RESULTS_DIR / '05_supervised_model_comparison.csv', index=False)
rf_importance.to_csv(RESULTS_DIR / '06_random_forest_feature_importance.csv', index=False)

print(f'Resultats exportés dans: {RESULTS_DIR}')
